In [1]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from merge_tables.db.tables import create_clean_account_name_macro

from pathlib import Path


MERGE_TABLES_DIR = Path('.').parent
DATA_DIR = MERGE_TABLES_DIR / "data"
OUTPUT_DIR = MERGE_TABLES_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
duck = connect_to_postgres_via_duckdb()
create_clean_account_name_macro(duck)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'
✓ Created clean_account_name macro


# get all customers from easybill

## where the Firma is null. Need to see what to do with them.

In [3]:
duck.sql(
    """
    select
        distinct on (Kundennummer)
        *
    from read_csv('/Users/adrienblanquer/Downloads/Contacts-Export-11_02_2026-15_23_01.csv')
    where Firma is null
    order by Firma
    """
)

┌────────────┬──────────────┬───────────────────┬───────────┬─────────┬────────────────────────┬─────────┬─────────────┬────────────────────────┬──────────────┬─────────┬────────────────────────────┬──────────────┬────────────┬────────────┬─────────────┬─────────────┬──────────┬───────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────────────────┬───────────┬───────────┬─────────┬──────────────┬──────────────────────────────┬─────────────────┬──────────┬─────────┬─────────┬────────────┬─────────────────────┬─────────────────┬────────────────┬──────────────┬──────────┬─────────────┬──────────────┬─────────────┬────────────────────────┬──────────┬──────────┬─────────┬────────────────┬─────────────────────┬───────────────────────┬────────────────────────┬────────────────────────────────────────┬─────────────────────────┬──────────────────────┬──────────────────────────┬──────────────────────────┬───────────────────────┬───────────────────────────────────┬────

# where firma is not null

In [4]:
duck.sql(
    """
    create or replace table easybill_active_clients as
    select
        distinct on (Kundennummer)
        *
    from read_csv('/Users/adrienblanquer/Downloads/Contacts-Export-11_02_2026-15_23_01.csv')
    where Firma is not null
    """
)

In [5]:
duck.sql("select * from easybill_active_clients")

┌────────────┬──────────────┬───────────────────┬───────────┬─────────┬────────────────────────┬─────────┬─────────┬───────────────────────┬──────────────┬─────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────┬──────────────┬───────────────────┬────────────┬─────────────┬─────────────┬──────────┬───────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────────────────┬─────────────────┬───────────┬─────────┬─────────────────┬───────────────────────────────────┬──────────────────────────────┬──────────┬────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬────────────┬─────────────────────┬─────────────────┬────────────────┬──────────────┬──────────┬─────────────┬──────────────┬─────────────┬────────────────────────┬─────────────────────────────┬──────────┬─────────────────────────────────────────────────────────────────────────────────────────────

# Zoho accounts

In [6]:
duck.sql(
    """
    create or replace table zoho_accounts as
    select * from read_csv('/Users/adrienblanquer/Downloads/Accounts_2026_02_03.csv', types={'Shipping Code': 'VARCHAR'})
    """
)

# Wochenliste

In [19]:
duck.sql(
    """
    create or replace table wochenliste as 
    select * 
    from read_xlsx('/Users/adrienblanquer/Downloads/2026_KW06_Wochenliste.xlsm', sheet='Kunden', header=True, range='B5:D998')
    """
)

# Match easybill active clients with zoho accounts using kundennummer

In [48]:
duck.sql(
    """
    with zoho_matching as (
        select 
            eb.Kundennummer,
            z."Accounts Name" as zoho_firm,
            z."Record ID" as zoho_id,
            z."BAS Standort" as zoho_standort,
            -- Determine the best match for each Kundennummer
            row_number() over (
                partition by eb.Kundennummer 
                order by 
                    (eb.Kundennummer::varchar = z.Kundennummer) desc,              -- Priority 1: ID
                    (eb.Firma = z."Accounts Name") desc,                           -- Priority 2: Name
                    jaro_winkler_similarity(eb.Firma, z."Accounts Name") desc      -- Priority 3: Fuzzy
            ) as rank
        from easybill_active_clients eb
        left join zoho_accounts z 
            on eb.Kundennummer::varchar = z.Kundennummer
            or eb.Firma = z."Accounts Name"
            or jaro_winkler_similarity(lower(eb.Firma), lower(z."Accounts Name")) > 0.95
    )
    select 
        eb.Kundennummer,
        eb.Firma as easybill_firm,
        zm.zoho_firm,
        case when w."ID Nummer" is not null then TRUE else FALSE end as wochenliste,
        w.Firmenname as wochenliste_firmenname,
        coalesce(w.Standort, zm.zoho_standort) as wochenliste_standort,
        zm.zoho_id
    from easybill_active_clients eb
    -- Join only with the top-ranked match from the CTE
    left join zoho_matching zm 
        on eb.Kundennummer = zm.Kundennummer 
        and zm.rank = 1
    left join wochenliste w 
        on eb.Kundennummer::varchar = left(w."ID Nummer"::varchar, 9)
    --where zm.zoho_firm is null
    order by eb.Firma
    """
).to_csv('easybill_zoho_wochenliste.csv')

In [71]:
duck.sql(
    """
    select * from pg.easybill.posten where "Kontakt: Kundennummer" = '130002302'
    """
)

┌───────────────────────┬────────────┬───────────────────────┬─────────────────────────────┬──────────────────┬─────────────┬───────────────────────────┬────────────────────────────┬───────────────────────────────────┬───────────────────┬──────────────────────┬────────────────────────────┬────────────────┬─────────────────┬─────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────────┬────────────────┬────────────────┬──────────────────┬────────────────────┬────────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┐
│ Kontakt: Kundennummer │ Posten: ID │ Posten: Artikelnummer │ Posten: Artikelbeschreibung │ Posten: Position │ Posten: Typ │ Posten: Einzelpreis Netto │ Posten: Einzelpreis Brutto │ Posten: Einkaufspreis pro Einheit │ Posten: Aufschlag │ Posten: Aufschlagtyp │ Posten: Gewinn pro Einheit │ Posten: Anzahl │ Posten: Einheit │ Posten: Nettobetrag │ Posten: USt. in Prozent │ Poste

In [74]:
duck.sql(
    """
    with firm_should_have_medisoft as (
        select 
            distinct on (Kundennummer)
            Kundennummer,
            case when sum(
                case 
                    when p."Posten: Artikelnummer" ilike '%-OMW%' then 1
                    else 0
                end
            ) > 0 then true else false end as should_have_medisoft_firm,

        from read_csv('easybill_zoho_wochenliste.csv') eb
        join pg.easybill.posten p
            on eb.Kundennummer = replace(p."Kontakt: Kundennummer", ' ', '')
            group by Kundennummer
    )
    select
         eb.*, fhm.should_have_medisoft_firm
    from read_csv('easybill_zoho_wochenliste.csv') eb
    left join firm_should_have_medisoft fhm
        on eb.Kundennummer = fhm.Kundennummer
    """
).to_csv('easybill_zoho_wochenliste_posten.csv')

# Add medisoft to the client list

In [87]:
duck.sql(
    """
    with medisoft_data as (
        select
            rec_id,
            kuerzel,
            name,
            pfad,
            case 
                when trim(split(pfad, '/')[2]) = 'Nicht Kunden' or trim(split(pfad, '/')[2]) = 'Kunden' then clean_account_name(split(pfad, '/')[3])
                else clean_account_name(split(pfad, '/')[2])
            end as clean_pfad_firm_name,
            clean_account_name(name) as clean_name,
            clean_account_name(kuerzel) as clean_kuerzel
        from pg.medisoft.table_firmenstruktur f
    ), easybill_zoho_data as (
        select 
            *,
            clean_account_name(easybill_firm) as clean_easybill_firm,
        from read_csv('easybill_zoho_wochenliste_posten.csv')
    )
    select 
        eb.* exclude (clean_easybill_firm),
        m.rec_id as medisoft_id,
        m.name,
        m.kuerzel,
        m.pfad as medisoft_path,
        m.clean_name,
        --jaro_winkler_similarity(eb.clean_easybill_firm, m.clean_pfad_firm_name) as sim
    from easybill_zoho_data eb
    left join medisoft_data m
        on jaro_winkler_similarity(eb.clean_easybill_firm, m.clean_pfad_firm_name) > 0.95
        or jaro_winkler_similarity(eb.clean_easybill_firm, m.clean_name) > 0.95
        or jaro_winkler_similarity(eb.clean_easybill_firm, m.clean_kuerzel) > 0.95
    --where Kundennummer = '130000200'
    order by easybill_firm
    """
).to_csv('easybill_zoho_wochenliste_posten_medisoft.csv')

In [88]:
duck.sql(
    """
    select count(distinct medisoft_id) from read_csv('easybill_zoho_wochenliste_posten_medisoft.csv')
    """
)

┌─────────────────────────────┐
│ count(DISTINCT medisoft_id) │
│            int64            │
├─────────────────────────────┤
│                        1666 │
└─────────────────────────────┘

In [94]:
duck.sql("select * from pg.medisoft.table_firmenstruktur where rec_id not in (select distinct medisoft_id from read_csv('easybill_zoho_wochenliste_posten_medisoft.csv') where medisoft_id is not null )")

┌──────────────────────────────────────┬───────────────────────────────────────────┬───────────────────────────────────────────┬───────────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────┬───────────────┬─────────────────┬───────────────┬───────────────┬───────────────┬────────────────┬──────────────────────────────────────────┬────────────┬─────────────────┬──────────┬─────────┬───────────┬────────────┬─────────────────┬─────────────────────────────────┬─────────┬───────────────────────┬────────────┬───────────────┬─────────┬──────────────────────┬─────────┬──────────────────┬──────────────┬─────────────────────────────┬─────────┬──────────┬─────────┬───────────────────┬────────────┬─────────────────────────┬────────────┬─────────────┬─────────────┬────────────────────┐
│                rec_id                │                  kuerzel                  │                   name                    │                           pfad                

In [ ]:
duck.sql(
    """
    select 
        regexp_extract(pfad, '^(.*?)\s*/', 1),
        regexp_extract(pfad, '/\s*(.*)$', 1) as pfad_firm_name,
        clean_account_name(pfad_firm_name) as clean_pfad_firm_name,
    from pg.medisoft.table_firmenstruktur
    where rec_id in ('00_9F600U7MB1', '00_9C100P9F9F')
    """
)

<>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/r3/svzsdlzn73xd8g34qr5rhw9h0000gq/T/ipykernel_3751/3449594985.py:4: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  regexp_extract(pfad, '^(.*?)\s*/', 1),


┌───────────────────────────────────────┬─────────────────────────────────────────────┬──────────────────────────────────────┐
│ regexp_extract(pfad, '^(.*?)\s*/', 1) │               pfad_firm_name                │         clean_pfad_firm_name         │
│                varchar                │                   varchar                   │               varchar                │
├───────────────────────────────────────┼─────────────────────────────────────────────┼──────────────────────────────────────┤
│ BSH Rostock                           │ Auf der Tenne Kita Buchenkopf Groß Lüsewitz │ aufdertennekitabuchenkopfgrolusewitz │
│ BSH Rostock                           │ „Auf der Tenne“ e.V.löschen!                │ aufdertenneevloschen                 │
└───────────────────────────────────────┴─────────────────────────────────────────────┴──────────────────────────────────────┘

In [ ]:
# Match easybill_active_clients to zoho_accounts: exact name first, then cleaned name if no exact match
duck.sql(
    """
    with cleaned_easybill_active_clients as (
        select
            *,
            clean_account_name(Firma) as clean_easybill_firm
        from easybill_active_clients
    ), cleaned_zoho_accounts as (
        select
            *,
            clean_account_name("Accounts Name") as clean_zoho_firm
        from zoho_accounts
    )
    select
        eb.Kundennummer,
        eb.Firma as easybill_firm,
        z."Accounts Name" as zoho_firm
    from cleaned_easybill_active_clients eb
    join cleaned_zoho_accounts z
        on eb.Firma = z."Accounts Name"
        or (
            eb.clean_easybill_firm = z.clean_zoho_firm
            and not exists (
                select 1 from cleaned_zoho_accounts z2
                where z2."Accounts Name" = eb.Firma
            )
        )
    where eb.Kundennummer not in (select Kundennummer from easybill_zoho_kundennummer_match)
    -- and eb.Kundennummer = '130000537'  -- optional: filter for one customer
    """
).show(max_rows=1000)

In [ ]:
duck.sql(
    """
    select
        c.*,
        matches.zoho_id as matched_zoho_id,
        matches.zoho_firm as matched_zoho_firm,
        matches.sim as similarity
    from read_csv('easybill_zoho_wochenliste.csv') c
    left join (
        with zoho_accounts_to_find as (
            select *
            from read_csv('easybill_zoho_wochenliste.csv')
            where zoho_firm is null
        )
        select
            zat.Kundennummer,
            za."Record ID" as zoho_id,
            za."Accounts Name" as zoho_firm,
            jaro_winkler_similarity(lower(zat.easybill_firm), lower(za."Accounts Name")) as sim
        from zoho_accounts_to_find zat
        join zoho_accounts za
            on jaro_winkler_similarity(lower(zat.easybill_firm), lower(za."Accounts Name")) > 0.95
        qualify row_number() over (partition by zat.Kundennummer order by sim desc) = 1
    ) matches using (Kundennummer)
    """
).to_csv('easybill_zoho_wochenliste_v2.csv')

In [60]:
duck.sql(
    """
    select * from pg.easybill.posten where "Kontakt: Kundennummer" = '130000380'
    """
)

┌───────────────────────┬────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────┬─────────────┬───────────────────────────┬────────────────────────────┬───────────────────────────────────┬───────────────────┬──────────────────────┬────────────────────────────┬────────────────┬─────────────────┬─────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────────┬────────────────┬────────────────┬──────────────────┬────────────────────┬────────────────────────┬────────────────────────┬──────────────────────┬────────────────────────┐
│ Kontakt: Kundennummer │ Posten: ID │ Posten: Artikelnummer │                                                                                            Posten: Artikelbeschreibung                

In [ ]:
duck.sql(
    """
    select * from 
    """
)